In [1]:
# --- setup (once) ---
from database.manager import DatabaseManager
from analysis.seasonality import Seasonality
from analysis.plotting import SeasonalityPlotter

db = DatabaseManager()
sznlty = Seasonality(db)
plotter = SeasonalityPlotter()

In [2]:
result = sznlty.outright_seasonality("CO", "Z", start_year=2019, end_year=2026, window_days=400)
plotter.plot_seasonality(result, title="CO Z — outright seasonality")

In [8]:
result = sznlty.expression_seasonality("G", "X26-Z26", start_year=2020, end_year=2026, window_days=400)
plotter.plot_seasonality(result, title="G X-Z spread — seasonality")

In [16]:
# --- minimal interactive dashboard for seasonality plotting ---
import ipywidgets as widgets
from IPython.display import display, clear_output

symbol_input = widgets.Text(value='CL', description='Symbol:')
expr_input = widgets.Text(value='Z26', description='Expression:')
from_year_input = widgets.IntText(value=2015, description='From Year:')
to_year_input = widgets.IntText(value=2026, description='To Year:')
window_input = widgets.IntText(value=400, description='Window Days:')
plot_button = widgets.Button(description='Plot Seasonality', button_style='success')
output = widgets.Output()

def on_plot_click(b):
    with output:
        clear_output(wait=True)
        symbol = symbol_input.value.strip().upper()
        expression = expr_input.value.strip().upper()
        start_year = from_year_input.value
        end_year = to_year_input.value
        window_days = window_input.value

        # Single-contract input (e.g. "Z26") behaves as an outright; anything with
        # +, -, or * is treated as a spread/fly expression.
        is_expression = any(op in expression for op in ['+', '-', '*']) and len(expression) > 3

        if is_expression:
            result = sznlty.expression_seasonality(
                symbol, expression, start_year=start_year, end_year=end_year, window_days=window_days
            )
            title = f"{symbol} {expression} — spread seasonality"
        else:
            letter = expression[0]
            result = sznlty.outright_seasonality(
                symbol, letter, start_year=start_year, end_year=end_year, window_days=window_days
            )
            title = f"{symbol} {letter} — outright seasonality"

        plotter.plot_seasonality(result, title=title)

plot_button.on_click(on_plot_click)

form = widgets.VBox([
    widgets.HBox([symbol_input, expr_input]),
    widgets.HBox([from_year_input, to_year_input, window_input]),
    plot_button,
    output
])
display(form)